# Notebook 1 · Naive vs. framework prompting

Same passage, two ways of asking a language model for reading-comprehension items:

1. **Naive:** "Write four multiple-choice questions about this passage."
2. **Framework:** one prompt per *cognitive category* — Focus and Retrieve (FR), Inference (INF),
   Interpret and Integrate (INT), Evaluate and Critique (CRIT). Each prompt says what the category
   is, gives the model a check to run before it outputs, and fixes the output layout (a JSON
   schema). Wrong options (distractors) are written in a second, category-aware step.

The point of the 15 minutes is to *see the difference*, not to build anything.

*Notebook by Rasmus Jensen (ETH Zurich), adapted for the fall school by Rudolf Debelak (EPFL). The pipeline follows Jensen (2025, 2026).*


## Setup

Three things to fill in: the endpoint, the key, and the two model names (one model writes the
passages, one writes and judges the items). On Colab, store the key once under the key icon on
the left (*Secrets*) as `AIG_API_KEY` and the endpoint as `AIG_BASE_URL`; the cell reads them
from there. Everything else in the notebook only uses the three small helpers defined below:

* `render(template, **values)` fills a prompt template — `{{level}}` in the template becomes the
  value of `level`.
* `ask(system, user, schema)` sends one system message and one user message to the model. If a
  `schema` (a pydantic class) is given, the server is asked to reply in that JSON layout and the
  reply comes back as an object of that class, so the rest of the code never parses text.
  Every call is stored in `cache/`; an identical call is replayed from disk, so a notebook that
  has been run once runs again without the endpoint (set `OFFLINE = True` to enforce that).
  Failed calls are retried; a reply that does not fit the schema is asked for once more.
* `ask_many(calls)` runs a list of such calls in parallel, because a pipeline step usually asks
  the same question about many items.


In [ ]:
%pip install -q openai pydantic jinja2 pandas

import json, re, os, time, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError
from jinja2 import Template
import pandas as pd

# --- endpoint, key, models -----------------------------------------------------------------
BASE_URL   = os.environ.get("AIG_BASE_URL", "https://YOUR-ENDPOINT/v1")   # the OpenAI-compatible endpoint
API_KEY    = os.environ.get("AIG_API_KEY", "EMPTY")
try:                                                     # on Colab: read both from the Secrets panel
    from google.colab import userdata
    API_KEY  = userdata.get("AIG_API_KEY") or API_KEY
    BASE_URL = userdata.get("AIG_BASE_URL") or BASE_URL
except Exception:
    pass
TEXT_MODEL = "swiss-ai/Apertus-70B-Instruct-2509"     # writes the passages
ITEM_MODEL = "Qwen/Qwen3-32B"                         # writes and judges the items (a thinking model)

# --- cache: every call is stored; identical calls are replayed ------------------------------
CACHE_DIR = Path("cache"); CACHE_DIR.mkdir(exist_ok=True)
OFFLINE   = False                                     # True: never call the endpoint, only replay

client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=240, max_retries=0)


def render(template, **values):
    return Template(template).render(**values)          # {{name}} in the template -> values[name]


def _parse(text, schema):
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)   # thinking models: drop the reasoning block
    text = re.sub(r"```(?:json)?", "", text).strip()              # and code fences
    if schema is None:
        return text
    data = json.loads(text)
    if isinstance(data, list) and len(data) == 1:                # some models wrap the object in a list
        data = data[0]
    return schema.model_validate(data)


def ask(system, user, schema=None, model=None, temperature=0.4, seed=None, think=None, max_tokens=4096, retries=3):
    """One system + one user message -> plain text, or an instance of `schema`."""
    payload = dict(model=model or ITEM_MODEL, temperature=temperature, max_tokens=max_tokens,
                   messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
    if schema is not None:                                        # ask the server for the schema's JSON layout
        payload["response_format"] = {"type": "json_schema",
                                      "json_schema": {"name": schema.__name__, "schema": schema.model_json_schema()}}
    if seed is not None:
        payload["seed"] = seed
    if think is not None:                                         # Qwen3-style thinking on/off (vLLM); ignored elsewhere
        payload["extra_body"] = {"chat_template_kwargs": {"enable_thinking": bool(think)}}

    path = CACHE_DIR / (hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()[:24] + ".json")
    for attempt in range(retries):
        cached = None
        if path.exists():
            try:
                cached = json.loads(path.read_text(encoding="utf-8"))["text"]
            except (OSError, ValueError):                         # half-written by a parallel call: treat as missing
                cached = None
        if cached is not None:
            text = cached
        elif OFFLINE:
            raise RuntimeError(f"OFFLINE and not in cache: {user[:80]!r}")
        else:
            try:
                reply = client.chat.completions.create(**payload)
                text = reply.choices[0].message.content or ""
            except Exception as e:                                # connection / server error: wait and retry
                time.sleep(3 * (attempt + 1))
                if attempt == retries - 1:
                    raise RuntimeError(f"endpoint failed {retries} times: {e}") from e
                continue
            tmp = path.with_name(path.name + f".{os.getpid()}.{id(payload)}.tmp")   # atomic write: parallel calls never see a half file
            tmp.write_text(json.dumps({"text": text, "payload": payload}, ensure_ascii=False), encoding="utf-8")
            os.replace(tmp, path)
        try:
            return _parse(text, schema)
        except (json.JSONDecodeError, ValidationError) as e:      # reply did not fit: drop it from the cache, ask again
            path.unlink(missing_ok=True)
            payload["temperature"] = 0.2
            if attempt == retries - 1:
                raise RuntimeError(f"reply did not match {schema.__name__}: {e}\n{text[:300]}") from e


def ask_many(calls, workers=6):                         # calls: list of dicts with ask()'s arguments
    with ThreadPoolExecutor(workers) as pool:
        return list(pool.map(lambda c: ask(**c), calls))


def need_file(name):
    """Make sure `name` exists in the session: on Colab, offer an upload dialog if it is missing."""
    if not Path(name).exists():
        try:
            from google.colab import files
            print(f"{name} not found - please upload it:")
            files.upload()
        except Exception:
            raise FileNotFoundError(f"{name} not found in the working directory")
    return name


In [ ]:
CATEGORIES = {
    'FR': {
        'name': 'Focus and Retrieve',
        'definition': 'The answer is stated in the text and the reader has to find it. Question and answer must be paraphrased, so that spotting a keyword is not enough.',
        'check': 'Does the question or the answer reuse the wording of the text? If yes, rephrase it.',
        'distractor': 'Each distractor must be something the text does NOT support anywhere.',
    },
    'INF': {
        'name': 'Inference',
        'definition': 'The answer is not stated in the text. The reader has to conclude it from clues the text gives.',
        'check': 'Could the answer be found by combining sentences of the text? If yes, it is not an inference - replace it.',
        'distractor': 'Each distractor must NOT be a valid conclusion from the text.',
    },
    'INT': {
        'name': 'Interpret and Integrate',
        'definition': 'The answer only follows from combining two different parts of the text. Neither part gives it alone.',
        'check': 'Could a reader who read only one of the two parts answer it? If yes, replace it.',
        'distractor': 'Each distractor is a wrong connection between the two parts, or a conclusion from one part only.',
    },
    'CRIT': {
        'name': 'Evaluate and Critique',
        'definition': 'The reader judges how the text is written: how effective an argument, an example, the structure or the ending is, and why.',
        'check': 'Is the answer a judgement about the writing rather than a summary of the content, and is the question not a yes/no question? If not, replace it.',
        'distractor': 'Each distractor must be a judgement the text clearly contradicts - not just a weaker judgement.',
    },
}

STEM_KEY_SYSTEM = """You write reading comprehension questions for English language tests."""

STEM_KEY_PROMPT = """Write {{n}} question(s) about the text, each with its correct answer only (no wrong options yet).

Category: {{name}}
{{definition}}
Before you output a question, check: {{check}}

Question: at most 150 characters. Answer: at most 90 characters. Language level: CEFR {{level}}.
Each question must target different information.

Text:
{{text}}"""

DISTRACTOR_SYSTEM = """You write multiple-choice options for reading tests."""

DISTRACTOR_PROMPT = """Write {{n}} wrong answer options (distractors) for this question.

Category: {{name}}. {{distractor}}
Every distractor must be clearly wrong for someone who read the text carefully, but tempting for someone who did not.
Match the correct answer in length and grammar. Do not write "all of the above" or "none of the above".
For each distractor, say briefly why it is wrong.

Text:
{{text}}

Question: {{question}}
Correct answer: {{answer}}"""

## 1. The passage

A B2 passage that ships with the notebook (fictional). Read it once — you will judge the items
against it. Notebook 2 shows how passages like this are generated.

In [ ]:
PASSAGE = {
    "id": "P_B2", "level": "B2", "title": "A Bridge or a Boat?",
    "text": """For as long as anyone can remember, the 380 residents of Tören, a low island off the northern coast, have reached the mainland by ferry. The crossing takes twenty-five minutes, runs six times a day in summer and three in winter, and is cancelled whenever the wind exceeds force seven. Last year, according to the harbour office, that happened on forty-one days. In November the regional government offered the islanders a choice they had never expected to face: a fixed bridge, paid for largely by the region, or a new, larger ferry with an ice-breaking hull.

The bridge would be 3.4 kilometres long and would open in 2031. Its supporters, led by the hotel owner Marit Lund, point to the school. Since the island's secondary school closed in 2014, teenagers have boarded on the mainland during the week, and Lund argues that a bridge would let families stay together and might persuade young couples to return. She also expects visitor numbers, currently about 30,000 a year, to double.

The ferry option has a quieter following. Its most visible advocate is the island's doctor, Anders Rask, who notes that the bridge plan includes a toll of eight euros per crossing, and that the region's own study expects at least two hundred cars a day in summer on an island that has, at present, a single narrow road and no car park. He rarely mentions what everybody knows: his surgery is the only one on the island, and the study counts the doctor's post among the services that a bridge would make "unnecessary to maintain locally".

The regional council will hold a consultative vote in May, but the final decision rests with the government. Lund has organised two public meetings, both well attended; Rask has organised none. Yet on the noticeboard outside the harbour office, where residents pin their comments, the ferry has been receiving the longer notes. The islanders, one senses, have discovered that the question is not really about transport at all, but about who they want to be in twenty years – a village that happens to be an island, or an island that happens to have a village.""",
}

print(PASSAGE["title"], f"({PASSAGE['level']}, {len(PASSAGE['text'].split())} words)\n")
print(PASSAGE["text"])

## 2. The naive prompt

One sentence, no definition of a good item, no output layout. `ask()` returns the reply as plain
text, so we just print it.

In [ ]:
NAIVE_PROMPT = """Write four multiple-choice questions about the passage below. Each question has four options (A-D) and exactly one correct answer; mark the correct answer.

Passage:
{{text}}"""

naive_reply = ask("You are a helpful assistant.", render(NAIVE_PROMPT, text=PASSAGE["text"]))
print(naive_reply)

## 3. The framework prompt

Print one of the four category prompts first (cell below) and look at its parts:

* **Category definition** — one sentence saying what the reader must do (e.g. INF: conclude
  something the text never says).
* **Check before output** — the "chain-of-thought" part: a question the model must ask itself
  about each item before it outputs it (e.g. INF: "could the answer be found by combining
  sentences of the text? then it is not an inference").
* **Constraints** — character limits, the CEFR level, no two questions on the same information.
* **Output schema** — the pydantic class `StemKeys` below. The server is asked to answer in exactly
  that layout, so the result is a list of `question`/`answer` pairs, not text to parse.

The framework writes the **question and the correct answer first** ("stem + key") and the
distractors later, because what makes a good distractor depends on the category.

`write_stem_keys(passage, cat, n)` does three things:
1. looks up the category's definition and check in `CATEGORIES`;
2. fills the prompt template with them, the level, the number of items and the text;
3. calls the model with the `StemKeys` schema and returns the parsed items.

In [ ]:
class StemKey(BaseModel):
    question: str = Field(description="The question, at most 150 characters.")
    answer: str = Field(description="The correct answer, at most 90 characters.")

class StemKeys(BaseModel):
    items: list[StemKey]


def write_stem_keys(passage, cat, n):
    c = CATEGORIES[cat]                                                     # 1. definition + check
    prompt = render(STEM_KEY_PROMPT, n=n, name=c["name"], definition=c["definition"],
                    check=c["check"], level=passage["level"], text=passage["text"])   # 2. fill the template
    return ask(STEM_KEY_SYSTEM, prompt, schema=StemKeys).items            # 3. call + parse


print(render(STEM_KEY_PROMPT, n=1, level="B2", text="<the passage>", **CATEGORIES["INF"]))

In [ ]:
stem_keys = {}                                  # category -> the one StemKey we asked for
for cat in CATEGORIES:
    stem_keys[cat] = write_stem_keys(PASSAGE, cat, n=1)[0]
    print(f"[{cat:4}] Q: {stem_keys[cat].question}\n       A: {stem_keys[cat].answer}\n")

## 4. Distractors, category-aware

The distractor prompt carries one extra line per category (`CATEGORIES[cat]["distractor"]`): an
FR distractor must not be supported by the text, an INF distractor must not be a valid
conclusion, an INT distractor is a wrong connection between the two parts, a CRIT distractor is a
judgement the text contradicts. The model also has to say *why* each distractor is wrong — that
sentence is a small validity check built into the output.

`write_distractors(passage, cat, item, n)` fills the template with the category's distractor
rule, the question and the correct answer, and returns the distractor texts. `assemble()` then
puts key and distractors into options A–D, **sorted alphabetically** so that the position of the
key carries no information.

In [ ]:
class Distractor(BaseModel):
    text: str
    why_wrong: str = Field(description="One sentence: why this option is wrong according to the text.")

class Distractors(BaseModel):
    distractors: list[Distractor]


def write_distractors(passage, cat, item, n):
    c = CATEGORIES[cat]
    prompt = render(DISTRACTOR_PROMPT, n=n, name=c["name"], distractor=c["distractor"],
                    text=passage["text"], question=item.question, answer=item.answer)
    return ask(DISTRACTOR_SYSTEM, prompt, schema=Distractors).distractors


def assemble(item, distractors):
    texts = sorted([item.answer] + distractors, key=str.lower)        # alphabetical: the key's position says nothing
    options = {letter: text for letter, text in zip("ABCD", texts)}
    correct = [L for L, t in options.items() if t == item.answer][0]
    return {"question": item.question, "options": options, "correct": correct}


def show(item, label=""):
    print(f"{label} {item['question']}")
    for L, text in item["options"].items():
        print(f"   {'*' if L == item['correct'] else ' '} {L}) {text}")
    print()


framework_items = {}
for cat, item in stem_keys.items():
    ds = write_distractors(PASSAGE, cat, item, n=3)
    framework_items[cat] = assemble(item, [d.text for d in ds])
    show(framework_items[cat], label=f"[{cat}]")
    for d in ds:
        print(f"      why wrong: {d.why_wrong}")
    print()

## 5. Compare

Look at the naive output and the four framework items side by side:

* **Categories** — how many of the naive questions just retrieve a fact? Is there one that needs a
  conclusion the text never states, or a judgement about the *writing*?
* **Copied wording** — does a naive question or answer reuse a phrase of the passage (keyword
  spotting)? The FR prompt forbids exactly that.
* **Distractors** — options of very different length, "all of the above", an option that is also
  defensible, an option nobody would choose.
* **Format** — the framework items are already structured data; the naive reply would have to be
  parsed by hand before Notebook 3 could screen it.

The cell below prints a compact side-by-side view and saves both for the discussion.

In [ ]:
rows = [{"source": "framework", "cat": cat, "question": it["question"],
         "key": it["options"][it["correct"]], "key_is_longest": len(it["options"][it["correct"]]) >= max(len(t) for t in it["options"].values())}
        for cat, it in framework_items.items()]
display(pd.DataFrame(rows))
print("\nNaive reply, for comparison:\n")
print(naive_reply)

with open("nb1_output.json", "w", encoding="utf-8") as f:
    json.dump({"passage": PASSAGE, "naive_reply": naive_reply, "framework_items": framework_items}, f, indent=2, ensure_ascii=False)
print("\nsaved nb1_output.json")